In [1]:
!pip install timm -q

In [2]:
# =============================================================================
# Diabetic Retinopathy Severity Grading — SUSTech-SYSU Dataset
# Single Kaggle P100 Training Script (Mixed Precision, Progressive Resolution)
# + GradCAM / Heatmap Visualisation
# =============================================================================

# ── Imports ──────────────────────────────────────────────────────────────────
import os
import gc
import csv
import copy
import random
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")


# =============================================================================
# SECTION 1 — CONFIGURATION
# =============================================================================

@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────────────────────
    img_dir: str = (
        "/kaggle/input/datasets/mariaherrerot/"
        "the-sustechsysu-dataset/SUSTech-SYSU grading dataset/"
        "SUSTech-SYSU grading dataset"
    )
    csv_path: str = (
        "/kaggle/input/datasets/mariaherrerot/"
        "the-sustechsysu-dataset/Labels.csv"
    )
    output_dir: str = "/kaggle/working"

    # ── Column names ──────────────────────────────────────────────────────────
    col_image: str = "Fundus_images"
    col_eye_side: str = "left_versus_right_eye(left_0_right_1)"
    col_intl: str = "DR_grade_International_Clinical_DR_Severity_Scale"    # primary
    col_aao: str = "DR_grade_American_Academy_of_Ophthalmology"            # aux 1
    col_scot: str = "DR_grade_Scottish_DR_grading_protocol"                # aux 2

    # ── Training ──────────────────────────────────────────────────────────────
    seed: int = 42
    base_batch_size: int = 16
    val_split: float = 0.20
    label_smoothing: float = 0.05
    weight_decay: float = 1e-4
    num_workers: int = 4
    pin_memory: bool = True

    # ── Loss weights ──────────────────────────────────────────────────────────
    loss_w_intl: float = 0.6
    loss_w_aao: float = 0.2
    loss_w_scot: float = 0.1
    loss_w_eye: float = 0.1

    # ── Progressive stages: (resolution, freeze_mode, epochs, lr) ─────────────
    stages: List[Tuple] = field(default_factory=lambda: [
        (224, "freeze",  5,  1e-3),
        (320, "partial", 10, 3e-4),
        (384, "full",    10, 1e-4),
        (448, "full",    5,  5e-5),
    ])

    # ── Early stopping ────────────────────────────────────────────────────────
    early_stop_patience: int = 5

    # ── Model ─────────────────────────────────────────────────────────────────
    model_name: str = "tf_efficientnetv2_s.in21k_ft_in1k"
    dropout: float = 0.4
    n_unfreeze_blocks: int = 3      # blocks unfrozen in "partial" mode

    # ── AMP ───────────────────────────────────────────────────────────────────
    use_amp: bool = True

    # ── Checkpoint filenames ──────────────────────────────────────────────────
    best_ckpt: str = "best_model.pth"
    log_csv: str = "training_log.csv"

    # ── GradCAM ───────────────────────────────────────────────────────────────
    gradcam_n_samples: int = 16      # number of val images to visualise
    gradcam_task: str = "intl"       # which head to differentiate ("intl"|"aao"|"scot"|"eye")
    gradcam_alpha: float = 0.45      # heatmap overlay opacity
    gradcam_cols: int = 4            # columns in the output grid


CFG = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG.seed)


def _worker_init_fn(worker_id: int) -> None:
    """Seed each DataLoader worker for reproducibility."""
    np.random.seed(CFG.seed + worker_id)
    random.seed(CFG.seed + worker_id)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# =============================================================================
# SECTION 2 — DATA INFRASTRUCTURE
# =============================================================================

# ── 2.1  Load & validate CSV ─────────────────────────────────────────────────

def load_csv(cfg: Config) -> pd.DataFrame:
    df = pd.read_csv(cfg.csv_path)
    required = [cfg.col_image, cfg.col_eye_side, cfg.col_intl, cfg.col_aao, cfg.col_scot]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing CSV columns: {missing}")

    # Drop rows with any NaN in label columns
    df = df.dropna(subset=required).reset_index(drop=True)

    # Cast label columns to int
    for col in [cfg.col_intl, cfg.col_aao, cfg.col_scot, cfg.col_eye_side]:
        df[col] = df[col].astype(int)

    print(f"[INFO] Dataset size after cleaning: {len(df)}")
    for col in [cfg.col_intl, cfg.col_aao, cfg.col_scot, cfg.col_eye_side]:
        print(f"  {col}: classes = {sorted(df[col].unique().tolist())}")

    return df


def infer_num_classes(df: pd.DataFrame, cfg: Config) -> Dict[str, int]:
    """Dynamically infer number of classes per head from the CSV values."""
    nc = {
        "intl": int(df[cfg.col_intl].max()) + 1,
        "aao":  int(df[cfg.col_aao].max()) + 1,
        "scot": int(df[cfg.col_scot].max()) + 1,
        "eye":  int(df[cfg.col_eye_side].max()) + 1,
    }
    print(f"[INFO] Inferred num_classes: {nc}")
    return nc


# ── 2.2  Fundus-specific preprocessing ───────────────────────────────────────

def crop_fundus(img: np.ndarray, tol: int = 7) -> np.ndarray:
    """Crop the black border around the fundus circle."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    coords = np.argwhere(mask)
    if coords.size == 0:
        return img
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    return img[y0:y1, x0:x1]


def apply_clahe_green(img: np.ndarray) -> np.ndarray:
    """Apply CLAHE to the green channel of a BGR image."""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    b, g, r = cv2.split(img)
    g_eq = clahe.apply(g)
    return cv2.merge([b, g_eq, r])


def load_and_preprocess(path: str, size: int) -> np.ndarray:
    """Load a fundus image through the full preprocessing pipeline."""
    img = cv2.imread(path)
    if img is None:
        return np.zeros((size, size, 3), dtype=np.uint8)
    img = crop_fundus(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LANCZOS4)
    img = apply_clahe_green(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img  # uint8 HxWxC RGB


# ── 2.3  Albumentations transforms ────────────────────────────────────────────

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def get_transforms(resolution: int, is_train: bool) -> A.Compose:
    if is_train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=20, p=0.7),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                               rotate_limit=15, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2,
                                       contrast_limit=0.2, p=0.5),
            A.CLAHE(clip_limit=2.0, p=0.3),
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
            A.ElasticTransform(alpha=1, sigma=50, p=0.3),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])


# ── 2.4  Dataset ──────────────────────────────────────────────────────────────

class DRDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        img_dir: str,
        resolution: int,
        transforms: A.Compose,
        cfg: Config,
    ):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.resolution = resolution
        self.transforms = transforms
        self.cfg = cfg

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_name = str(row[self.cfg.col_image])
        if not img_name.lower().endswith(".jpg"):
            img_name = img_name + ".jpg"
        img_path = os.path.join(self.img_dir, img_name)

        img = load_and_preprocess(img_path, self.resolution)
        augmented = self.transforms(image=img)
        image = augmented["image"]  # CxHxW float32 tensor

        labels = {
            "intl": torch.tensor(int(row[self.cfg.col_intl]), dtype=torch.long),
            "aao":  torch.tensor(int(row[self.cfg.col_aao]),  dtype=torch.long),
            "scot": torch.tensor(int(row[self.cfg.col_scot]), dtype=torch.long),
            "eye":  torch.tensor(int(row[self.cfg.col_eye_side]), dtype=torch.long),
        }
        return image, labels


# ── 2.5  Stratified split ─────────────────────────────────────────────────────

def stratified_split(
    df: pd.DataFrame, cfg: Config
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    sss = StratifiedShuffleSplit(
        n_splits=1, test_size=cfg.val_split, random_state=cfg.seed
    )
    train_idx, val_idx = next(sss.split(df, df[cfg.col_intl]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)
    print(f"[INFO] Train: {len(train_df)} | Val: {len(val_df)}")
    return train_df, val_df


# ── 2.6  DataLoader builder ───────────────────────────────────────────────────

def build_dataloaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    resolution: int,
    cfg: Config,
) -> Tuple[DataLoader, DataLoader]:
    train_ds = DRDataset(
        train_df, cfg.img_dir, resolution,
        get_transforms(resolution, is_train=True), cfg
    )
    val_ds = DRDataset(
        val_df, cfg.img_dir, resolution,
        get_transforms(resolution, is_train=False), cfg
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.base_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        drop_last=True,
        persistent_workers=(cfg.num_workers > 0),
        worker_init_fn=_worker_init_fn,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.base_batch_size * 2,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        drop_last=False,
        persistent_workers=(cfg.num_workers > 0),
        worker_init_fn=_worker_init_fn,
    )
    return train_loader, val_loader


# =============================================================================
# SECTION 3 — MODEL ARCHITECTURE
# =============================================================================

class MultiTaskDRModel(nn.Module):
    def __init__(self, model_name: str, num_classes: Dict[str, int], dropout: float):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,       # remove classifier
            global_pool="avg",   # Global Average Pooling
        )
        feature_dim = self.backbone.num_features

        # Shared neck: BN → Dropout
        self.neck = nn.Sequential(
            nn.BatchNorm1d(feature_dim),
            nn.Dropout(p=dropout),
        )

        # Four independent classification heads
        self.head_intl = nn.Linear(feature_dim, num_classes["intl"])
        self.head_aao  = nn.Linear(feature_dim, num_classes["aao"])
        self.head_scot = nn.Linear(feature_dim, num_classes["scot"])
        self.head_eye  = nn.Linear(feature_dim, num_classes["eye"])

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        feat = self.backbone(x)     # (B, feature_dim)
        feat = self.neck(feat)      # (B, feature_dim)
        return {
            "intl": self.head_intl(feat),
            "aao":  self.head_aao(feat),
            "scot": self.head_scot(feat),
            "eye":  self.head_eye(feat),
        }


# ── Freeze helpers ────────────────────────────────────────────────────────────

def freeze_encoder(model: MultiTaskDRModel) -> None:
    for param in model.backbone.parameters():
        param.requires_grad = False
    print("[INFO] Encoder frozen.")


def unfreeze_last_blocks(model: MultiTaskDRModel, n_blocks: int = 3) -> None:
    """Unfreeze the last n_blocks of the EfficientNetV2 backbone."""
    for param in model.backbone.parameters():
        param.requires_grad = False

    blocks = model.backbone.blocks
    n_stages = len(blocks)
    stages_to_unfreeze = blocks[max(0, n_stages - n_blocks):]
    for stage in stages_to_unfreeze:
        for param in stage.parameters():
            param.requires_grad = True

    for attr in ["conv_head", "bn2"]:
        if hasattr(model.backbone, attr):
            for param in getattr(model.backbone, attr).parameters():
                param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[INFO] Partial unfreeze: {trainable:,} trainable parameters.")


def unfreeze_all(model: MultiTaskDRModel) -> None:
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[INFO] Full unfreeze: {trainable:,} trainable parameters.")


# =============================================================================
# SECTION 4 — LOSS & METRICS
# =============================================================================

def compute_class_weights_tensor(
    df: pd.DataFrame, col: str, num_classes: int, device: torch.device
) -> torch.Tensor:
    present_classes = sorted(df[col].unique())
    try:
        weights = compute_class_weight(
            class_weight="balanced",
            classes=np.array(present_classes),
            y=df[col].values,
        )
        weight_tensor = torch.ones(num_classes, dtype=torch.float32)
        for cls, w in zip(present_classes, weights):
            weight_tensor[cls] = float(w)
    except Exception:
        weight_tensor = torch.ones(num_classes, dtype=torch.float32)
    return weight_tensor.to(device)


class MultiTaskLoss(nn.Module):
    def __init__(
        self,
        class_weights: Dict[str, torch.Tensor],
        label_smoothing: float,
        loss_weights: Tuple[float, float, float, float],
    ):
        super().__init__()
        self.w_intl, self.w_aao, self.w_scot, self.w_eye = loss_weights
        self.ce_intl = nn.CrossEntropyLoss(
            weight=class_weights["intl"], label_smoothing=label_smoothing
        )
        self.ce_aao = nn.CrossEntropyLoss(
            weight=class_weights["aao"],  label_smoothing=label_smoothing
        )
        self.ce_scot = nn.CrossEntropyLoss(
            weight=class_weights["scot"], label_smoothing=label_smoothing
        )
        self.ce_eye = nn.CrossEntropyLoss(
            weight=class_weights["eye"],  label_smoothing=label_smoothing
        )

    def forward(
        self,
        logits: Dict[str, torch.Tensor],
        labels: Dict[str, torch.Tensor],
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        l_intl = self.ce_intl(logits["intl"], labels["intl"])
        l_aao  = self.ce_aao(logits["aao"],   labels["aao"])
        l_scot = self.ce_scot(logits["scot"],  labels["scot"])
        l_eye  = self.ce_eye(logits["eye"],    labels["eye"])

        total = (
            self.w_intl * l_intl
            + self.w_aao  * l_aao
            + self.w_scot * l_scot
            + self.w_eye  * l_eye
        )
        per_task = {
            "intl": l_intl.item(),
            "aao":  l_aao.item(),
            "scot": l_scot.item(),
            "eye":  l_eye.item(),
        }
        return total, per_task


def compute_metrics(
    y_true: np.ndarray, y_pred: np.ndarray
) -> Dict[str, object]:
    acc   = accuracy_score(y_true, y_pred)
    try:
        qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    except Exception:
        qwk = 0.0
    f1    = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm    = confusion_matrix(y_true, y_pred)
    return {"accuracy": acc, "qwk": qwk, "f1": f1, "cm": cm}


# =============================================================================
# SECTION 5 — TRAINING INFRASTRUCTURE
# =============================================================================

def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: MultiTaskLoss,
    scaler: GradScaler,
    device: torch.device,
    use_amp: bool,
) -> float:
    model.train()
    total_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = {k: v.to(device, non_blocking=True) for k, v in labels.items()}

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            logits = model(images)
            loss, _ = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: MultiTaskLoss,
    device: torch.device,
    use_amp: bool,
) -> Tuple[float, Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    model.eval()
    total_loss = 0.0
    all_preds  = {k: [] for k in ["intl", "aao", "scot", "eye"]}
    all_labels = {k: [] for k in ["intl", "aao", "scot", "eye"]}

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = {k: v.to(device, non_blocking=True) for k, v in labels.items()}

            with autocast("cuda", enabled=use_amp):
                logits = model(images)
                loss, _ = criterion(logits, labels)

            total_loss += loss.item()

            for key in all_preds:
                preds = logits[key].argmax(dim=1).cpu().numpy()
                all_preds[key].extend(preds.tolist())
                all_labels[key].extend(labels[key].cpu().numpy().tolist())

    all_preds  = {k: np.array(v) for k, v in all_preds.items()}
    all_labels = {k: np.array(v) for k, v in all_labels.items()}
    return total_loss / len(loader), all_preds, all_labels


class EarlyStopping:
    def __init__(self, patience: int, output_dir: str, ckpt_name: str):
        self.patience = patience
        self.counter = 0
        self.best_score: float = -float("inf")
        self.best_state: Optional[dict] = None
        self.ckpt_path = os.path.join(output_dir, ckpt_name)

    def step(self, score: float, model: nn.Module) -> bool:
        if score > self.best_score:
            self.best_score = score
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
            torch.save(self.best_state, self.ckpt_path)
            print(f"  [Checkpoint] Best QWK = {score:.4f} saved.")
        else:
            self.counter += 1
            print(f"  [EarlyStopping] {self.counter}/{self.patience} (best={self.best_score:.4f})")
            if self.counter >= self.patience:
                return True
        return False

    def restore(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)
            print(f"[INFO] Restored best weights (QWK={self.best_score:.4f}).")


class CSVLogger:
    def __init__(self, path: str):
        self.path = path
        self.fieldnames = [
            "stage", "epoch", "train_loss", "val_loss",
            "val_qwk", "val_acc", "val_f1",
        ]
        with open(self.path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            writer.writeheader()

    def log(self, row: dict) -> None:
        with open(self.path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            writer.writerow(row)


# =============================================================================
# SECTION 6 — PROGRESSIVE TRAINING LOOP
# =============================================================================

def run_stage(
    stage_idx: int,
    resolution: int,
    freeze_mode: str,
    n_epochs: int,
    lr: float,
    model: MultiTaskDRModel,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    criterion: MultiTaskLoss,
    cfg: Config,
    logger: CSVLogger,
    early_stopper: EarlyStopping,
    scaler: GradScaler,
) -> None:
    print(f"\n{'='*60}")
    print(f" STAGE {stage_idx+1} | res={resolution} | mode={freeze_mode} | "
          f"epochs={n_epochs} | lr={lr}")
    print(f"{'='*60}")

    if freeze_mode == "freeze":
        freeze_encoder(model)
    elif freeze_mode == "partial":
        unfreeze_last_blocks(model, cfg.n_unfreeze_blocks)
    else:
        unfreeze_all(model)

    train_loader, val_loader = build_dataloaders(train_df, val_df, resolution, cfg)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params, lr=lr, weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=lr / 10.0
    )

    early_stopper.counter = 0

    for epoch in range(1, n_epochs + 1):
        train_loss = train_epoch(
            model, train_loader, optimizer, criterion,
            scaler, DEVICE, cfg.use_amp
        )
        val_loss, val_preds, val_labels = validate_epoch(
            model, val_loader, criterion, DEVICE, cfg.use_amp
        )
        scheduler.step()

        metrics      = compute_metrics(val_labels["intl"], val_preds["intl"])
        aux_acc_aao  = accuracy_score(val_labels["aao"],  val_preds["aao"])
        aux_acc_scot = accuracy_score(val_labels["scot"], val_preds["scot"])
        current_lr   = optimizer.param_groups[0]["lr"]

        print(
            f"  Epoch {epoch:02d}/{n_epochs} | lr={current_lr:.2e} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"QWK={metrics['qwk']:.4f} | acc={metrics['accuracy']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"aao_acc={aux_acc_aao:.3f} | scot_acc={aux_acc_scot:.3f}"
        )

        logger.log({
            "stage":      stage_idx + 1,
            "epoch":      epoch,
            "train_loss": round(train_loss, 6),
            "val_loss":   round(val_loss, 6),
            "val_qwk":    round(metrics["qwk"], 6),
            "val_acc":    round(metrics["accuracy"], 6),
            "val_f1":     round(metrics["f1"], 6),
        })

        should_stop = early_stopper.step(metrics["qwk"], model)
        if should_stop:
            print(f"  [EarlyStopping] Triggered at epoch {epoch}.")
            break

    early_stopper.restore(model)

    del train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_all_stages(
    model: MultiTaskDRModel,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    criterion: MultiTaskLoss,
    cfg: Config,
) -> None:
    log_path = os.path.join(cfg.output_dir, cfg.log_csv)
    logger   = CSVLogger(log_path)
    scaler   = GradScaler("cuda", enabled=cfg.use_amp)
    early_stopper = EarlyStopping(
        patience=cfg.early_stop_patience,
        output_dir=cfg.output_dir,
        ckpt_name=cfg.best_ckpt,
    )

    for stage_idx, (resolution, freeze_mode, n_epochs, lr) in enumerate(cfg.stages):
        run_stage(
            stage_idx=stage_idx,
            resolution=resolution,
            freeze_mode=freeze_mode,
            n_epochs=n_epochs,
            lr=lr,
            model=model,
            train_df=train_df,
            val_df=val_df,
            criterion=criterion,
            cfg=cfg,
            logger=logger,
            early_stopper=early_stopper,
            scaler=scaler,
        )

    print("\n[INFO] All stages complete. Best checkpoint saved at:", early_stopper.ckpt_path)
    print(f"[INFO] Overall best validation QWK: {early_stopper.best_score:.4f}")


# =============================================================================
# SECTION 7 — TEST TIME AUGMENTATION (TTA) + FINAL EVALUATION
# =============================================================================

def _tta_transforms(resolution: int) -> List[A.Compose]:
    """Returns 5 distinct TTA transforms."""
    norm = [A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return [
        A.Compose(norm),
        A.Compose([A.HorizontalFlip(p=1.0)] + norm),
        A.Compose([A.Rotate(limit=(10, 10), p=1.0)] + norm),
        A.Compose([A.Rotate(limit=(-10, -10), p=1.0)] + norm),
        A.Compose([
            A.CenterCrop(
                height=int(resolution * 0.9),
                width=int(resolution * 0.9),
                p=1.0
            ),
            A.Resize(resolution, resolution),
        ] + norm),
    ]


def tta_predict(
    model: nn.Module,
    val_df: pd.DataFrame,
    resolution: int,
    cfg: Config,
    device: torch.device,
    n_views: int = 5,
) -> Dict[str, np.ndarray]:
    model.eval()
    tta_tfms = _tta_transforms(resolution)[:n_views]

    imgs_raw: List[np.ndarray] = []
    for idx in range(len(val_df)):
        row = val_df.iloc[idx]
        img_name = str(row[cfg.col_image])
        if not img_name.lower().endswith(".jpg"):
            img_name += ".jpg"
        path = os.path.join(cfg.img_dir, img_name)
        imgs_raw.append(load_and_preprocess(path, resolution))

    num_samples = len(imgs_raw)
    all_probs: Dict[str, Optional[np.ndarray]] = {
        k: None for k in ["intl", "aao", "scot", "eye"]
    }

    for tfm in tta_tfms:
        batch_size = cfg.base_batch_size * 2
        probs_this_view: Dict[str, List[np.ndarray]] = {
            k: [] for k in all_probs
        }

        for start in range(0, num_samples, batch_size):
            batch_imgs = []
            for img in imgs_raw[start: start + batch_size]:
                aug = tfm(image=img)
                batch_imgs.append(aug["image"])
            tensor = torch.stack(batch_imgs).to(device, non_blocking=True)

            with torch.no_grad():
                with autocast("cuda", enabled=cfg.use_amp):
                    logits = model(tensor)

            for key in all_probs:
                probs_batch = F.softmax(logits[key], dim=1).cpu().numpy()
                probs_this_view[key].append(probs_batch)

        for key in all_probs:
            view_probs = np.concatenate(probs_this_view[key], axis=0)
            if all_probs[key] is None:
                all_probs[key] = view_probs
            else:
                all_probs[key] = all_probs[key] + view_probs

    final_preds: Dict[str, np.ndarray] = {}
    for key in all_probs:
        avg_probs = all_probs[key] / n_views
        final_preds[key] = avg_probs.argmax(axis=1)

    return final_preds


def final_evaluation(
    model: nn.Module,
    val_df: pd.DataFrame,
    cfg: Config,
    device: torch.device,
) -> Dict[str, object]:
    final_resolution = cfg.stages[-1][0]
    print(f"\n[INFO] Running TTA final evaluation at resolution {final_resolution}...")

    tta_preds = tta_predict(model, val_df, final_resolution, cfg, device)
    y_true    = val_df[cfg.col_intl].values.astype(int)
    y_pred    = tta_preds["intl"]

    metrics = compute_metrics(y_true, y_pred)

    print("\n" + "="*60)
    print(" FINAL EVALUATION RESULTS (Primary Task — International DR Grade)")
    print("="*60)
    print(f"  Accuracy               : {metrics['accuracy']:.4f}")
    print(f"  Quadratic Weighted Kappa: {metrics['qwk']:.4f}")
    print(f"  Macro F1-Score         : {metrics['f1']:.4f}")
    print("\n  Confusion Matrix:")
    print(metrics["cm"])
    print("="*60)

    return metrics


# =============================================================================
# SECTION 8 — GRADCAM
# =============================================================================

class GradCAM:
    """
    Gradient-weighted Class Activation Mapping for MultiTaskDRModel.

    Hooks into a target convolutional layer, runs a forward pass,
    then back-propagates the score of the predicted (or specified) class
    through the chosen head to produce a spatial importance map.

    Usage
    -----
    cam = GradCAM(model, target_layer=model.backbone.blocks[-1][-1])
    heatmap = cam(image_tensor, task="intl", class_idx=None)
    cam.remove_hooks()
    """

    def __init__(self, model: MultiTaskDRModel, target_layer: nn.Module):
        self.model = model
        self._activations: Optional[torch.Tensor] = None
        self._gradients: Optional[torch.Tensor] = None

        # Register forward hook to capture feature maps
        self._fwd_hook = target_layer.register_forward_hook(self._save_activation)
        # Register backward hook to capture gradients
        self._bwd_hook = target_layer.register_full_backward_hook(self._save_gradient)

    # ── Hook callbacks ────────────────────────────────────────────────────────

    def _save_activation(
        self,
        module: nn.Module,
        input: Tuple,
        output: torch.Tensor,
    ) -> None:
        # output shape: (B, C, H, W)
        self._activations = output.detach()

    def _save_gradient(
        self,
        module: nn.Module,
        grad_input: Tuple,
        grad_output: Tuple,
    ) -> None:
        # grad_output[0] shape: (B, C, H, W)
        self._gradients = grad_output[0].detach()

    # ── Core computation ──────────────────────────────────────────────────────

    def __call__(
        self,
        image_tensor: torch.Tensor,   # (1, C, H, W) on the correct device
        task: str = "intl",
        class_idx: Optional[int] = None,
    ) -> np.ndarray:
        """
        Returns a normalised GradCAM heatmap as a float32 array in [0, 1]
        with shape (H_img, W_img) matching the input spatial dimensions.
        """
        self.model.eval()

        # ── Forward ───────────────────────────────────────────────────────────
        image_tensor = image_tensor.requires_grad_(False)  # hooks handle grads
        logits = self.model(image_tensor)                  # Dict[str, Tensor]
        head_logits = logits[task]                         # (1, num_classes)

        # Choose the target class (predicted class by default)
        if class_idx is None:
            class_idx = head_logits.argmax(dim=1).item()

        # ── Backward ──────────────────────────────────────────────────────────
        self.model.zero_grad()
        score = head_logits[0, class_idx]
        score.backward()

        # ── Weight feature maps by global-average-pooled gradients ────────────
        # gradients: (1, C, H, W)  →  weights: (C,)
        weights = self._gradients[0].mean(dim=(1, 2))   # (C,)
        # activations: (1, C, H, W)  →  (C, H, W)
        activations = self._activations[0]              # (C, H, W)

        # Weighted combination: sum over channels
        cam = torch.einsum("c,chw->hw", weights, activations)  # (H, W)

        # ReLU: keep only positive contributions
        cam = F.relu(cam)

        # Normalise to [0, 1]
        cam_min = cam.min()
        cam_max = cam.max()
        if (cam_max - cam_min) > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = torch.zeros_like(cam)

        return cam.cpu().numpy()   # (H_feat, W_feat) in [0, 1]

    def remove_hooks(self) -> None:
        """Must be called after use to avoid memory leaks."""
        self._fwd_hook.remove()
        self._bwd_hook.remove()


# ── Helper: locate the target layer in the backbone ──────────────────────────

def _get_gradcam_target_layer(model: MultiTaskDRModel) -> nn.Module:
    """
    Returns the last convolutional block of the EfficientNetV2 backbone.
    For tf_efficientnetv2_s this is model.backbone.blocks[-1][-1].
    Falls back to conv_head if blocks are unavailable.
    """
    backbone = model.backbone
    if hasattr(backbone, "blocks") and len(backbone.blocks) > 0:
        last_stage = backbone.blocks[-1]
        # last_stage may be a Sequential or a list-like ModuleList
        if hasattr(last_stage, "__len__") and len(last_stage) > 0:
            return last_stage[-1]
        return last_stage
    if hasattr(backbone, "conv_head"):
        return backbone.conv_head
    # Generic fallback: last module with weight
    for module in reversed(list(backbone.modules())):
        if isinstance(module, nn.Conv2d):
            return module
    raise RuntimeError("Could not locate a suitable GradCAM target layer.")


# ── Overlay helper ────────────────────────────────────────────────────────────

def overlay_heatmap(
    rgb_image: np.ndarray,   # uint8 (H, W, 3)
    cam: np.ndarray,         # float32 (h, w) in [0, 1]
    alpha: float = 0.45,
    colormap: int = cv2.COLORMAP_JET,
) -> np.ndarray:
    """
    Resize cam to match rgb_image, apply a colormap, and blend.
    Returns a uint8 (H, W, 3) RGB overlay.
    """
    h, w = rgb_image.shape[:2]

    # Resize CAM to image resolution
    cam_resized = cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR)

    # Apply colormap (JET: blue=low, red=high)
    cam_uint8   = np.uint8(255 * cam_resized)
    cam_colored = cv2.applyColorMap(cam_uint8, colormap)           # BGR
    cam_rgb     = cv2.cvtColor(cam_colored, cv2.COLOR_BGR2RGB)     # RGB

    # Weighted blend
    blended = (
        (1.0 - alpha) * rgb_image.astype(np.float32)
        + alpha * cam_rgb.astype(np.float32)
    ).clip(0, 255).astype(np.uint8)

    return blended


# ── Main GradCAM generation function ─────────────────────────────────────────

def generate_gradcam_heatmaps(
    model: MultiTaskDRModel,
    val_df: pd.DataFrame,
    cfg: Config,
    device: torch.device,
    n_samples: Optional[int] = None,
    task: Optional[str] = None,
) -> None:
    """
    Generates GradCAM heatmaps for a random subset of validation images
    and saves:
      • gradcam_grid.png  — mosaic of (original | heatmap | overlay) triplets
      • gradcam/          — individual overlay PNGs for each sample

    Parameters
    ----------
    n_samples : number of images to visualise (default: cfg.gradcam_n_samples)
    task      : which classification head to explain (default: cfg.gradcam_task)
    """
    n_samples = n_samples or cfg.gradcam_n_samples
    task      = task      or cfg.gradcam_task
    resolution = cfg.stages[-1][0]   # use final-stage resolution

    print(f"\n[GradCAM] Generating heatmaps for {n_samples} images (task='{task}', "
          f"res={resolution})...")

    # Sample indices (stratified by DR grade for coverage)
    sample_df = (
        val_df.groupby(cfg.col_intl, group_keys=False)
              .apply(lambda g: g.sample(
                  min(len(g), max(1, n_samples // val_df[cfg.col_intl].nunique())),
                  random_state=cfg.seed,
              ))
              .head(n_samples)
              .reset_index(drop=True)
    )

    # Preprocessing transform (no augmentation, just normalise)
    tfm = get_transforms(resolution, is_train=False)

    # Build GradCAM with hooks on the last backbone block
    target_layer = _get_gradcam_target_layer(model)
    print(f"[GradCAM] Target layer: {target_layer.__class__.__name__}")
    cam_extractor = GradCAM(model, target_layer)

    # Output dirs
    grid_dir = cfg.output_dir
    per_img_dir = os.path.join(cfg.output_dir, "gradcam")
    os.makedirs(per_img_dir, exist_ok=True)

    # ── Collect triplets (original, heatmap-only, overlay) ────────────────────
    cols   = cfg.gradcam_cols
    n_imgs = len(sample_df)
    rows   = n_imgs

    fig_w  = cols * 3 * 3     # 3 panels per sample × 3 inches each × cols
    fig_h  = rows * 3
    fig, axes = plt.subplots(rows, cols * 3, figsize=(fig_w, fig_h))
    if rows == 1:
        axes = axes[np.newaxis, :]  # ensure 2-D indexing

    DR_GRADE_LABELS = {0: "No DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Prolif."}

    for sample_i, (_, row) in enumerate(sample_df.iterrows()):
        # ── Load raw image ─────────────────────────────────────────────────────
        img_name = str(row[cfg.col_image])
        if not img_name.lower().endswith(".jpg"):
            img_name += ".jpg"
        img_path = os.path.join(cfg.img_dir, img_name)
        rgb = load_and_preprocess(img_path, resolution)   # uint8 (H, W, 3)

        # ── Build normalised tensor ────────────────────────────────────────────
        aug    = tfm(image=rgb)
        tensor = aug["image"].unsqueeze(0).to(device)     # (1, C, H, W)
        tensor.requires_grad_(True)

        # ── Run GradCAM ────────────────────────────────────────────────────────
        cam_map = cam_extractor(tensor, task=task, class_idx=None)  # (h, w) [0,1]

        # Predicted & true grade
        with torch.no_grad():
            logits   = model(tensor)
        pred_cls = int(logits[task].argmax(dim=1).item())
        true_cls = int(row[cfg.col_intl])

        # ── Build overlay and colour-only heatmap ─────────────────────────────
        overlay  = overlay_heatmap(rgb, cam_map, alpha=cfg.gradcam_alpha)
        cam_vis  = cv2.applyColorMap(np.uint8(255 * cv2.resize(
            cam_map, (resolution, resolution), interpolation=cv2.INTER_LINEAR
        )), cv2.COLORMAP_JET)
        cam_vis  = cv2.cvtColor(cam_vis, cv2.COLOR_BGR2RGB)

        # ── Save individual overlay ────────────────────────────────────────────
        save_name = f"gradcam_{sample_i:03d}_true{true_cls}_pred{pred_cls}.png"
        cv2.imwrite(
            os.path.join(per_img_dir, save_name),
            cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR),
        )

        # ── Place in grid ──────────────────────────────────────────────────────
        row_i = sample_i
        col_start = 0   # single-column layout per row (orig | cam | overlay)

        true_lbl = DR_GRADE_LABELS.get(true_cls, str(true_cls))
        pred_lbl = DR_GRADE_LABELS.get(pred_cls, str(pred_cls))
        correct  = "✓" if pred_cls == true_cls else "✗"
        title_colour = "green" if pred_cls == true_cls else "red"

        # Original
        ax0 = axes[row_i, col_start]
        ax0.imshow(rgb)
        ax0.set_title(f"Original\nTrue: {true_lbl}", fontsize=7, pad=2)
        ax0.axis("off")

        # Pure CAM
        ax1 = axes[row_i, col_start + 1]
        im  = ax1.imshow(cam_map, cmap="jet", vmin=0, vmax=1)
        ax1.set_title(f"GradCAM\n(task: {task})", fontsize=7, pad=2)
        ax1.axis("off")
        plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

        # Overlay
        ax2 = axes[row_i, col_start + 2]
        ax2.imshow(overlay)
        ax2.set_title(f"Overlay  {correct}\nPred: {pred_lbl}",
                      fontsize=7, pad=2, color=title_colour)
        ax2.axis("off")

        # Hide unused columns if cols > 1
        for c in range(3, cols * 3):
            axes[row_i, c].axis("off")

    plt.suptitle(
        f"GradCAM — task: {task}  |  resolution: {resolution}px\n"
        f"(green title = correct prediction, red = incorrect)",
        fontsize=10, y=1.002,
    )
    plt.tight_layout()

    grid_path = os.path.join(grid_dir, "gradcam_grid.png")
    plt.savefig(grid_path, dpi=120, bbox_inches="tight")
    plt.close()

    cam_extractor.remove_hooks()

    print(f"[GradCAM] Grid saved  → {grid_path}")
    print(f"[GradCAM] Per-image overlays → {per_img_dir}/")


# ── Multi-head GradCAM comparison (all 4 tasks on one image) ──────────────────

def gradcam_all_heads(
    model: MultiTaskDRModel,
    val_df: pd.DataFrame,
    cfg: Config,
    device: torch.device,
    n_samples: int = 4,
) -> None:
    """
    For n_samples images, shows GradCAM side-by-side for all 4 task heads.
    Saved as gradcam_multihead.png.
    """
    resolution = cfg.stages[-1][0]
    tfm        = get_transforms(resolution, is_train=False)
    tasks      = ["intl", "aao", "scot", "eye"]
    task_names = {
        "intl": "Intl. DR Grade",
        "aao":  "AAO Grade",
        "scot": "Scottish Grade",
        "eye":  "Eye Side",
    }

    sample_df = val_df.sample(
        min(n_samples, len(val_df)), random_state=cfg.seed
    ).reset_index(drop=True)

    target_layer  = _get_gradcam_target_layer(model)
    cam_extractor = GradCAM(model, target_layer)

    n_rows = len(sample_df)
    n_cols = 1 + len(tasks)   # original + one per task

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 2.8, n_rows * 2.8),
    )
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for r, (_, row) in enumerate(sample_df.iterrows()):
        img_name = str(row[cfg.col_image])
        if not img_name.lower().endswith(".jpg"):
            img_name += ".jpg"
        rgb = load_and_preprocess(
            os.path.join(cfg.img_dir, img_name), resolution
        )
        aug    = tfm(image=rgb)
        tensor = aug["image"].unsqueeze(0).to(device)
        tensor.requires_grad_(True)

        true_grade = int(row[cfg.col_intl])

        # Column 0: original
        axes[r, 0].imshow(rgb)
        axes[r, 0].set_title(f"Original\nDR={true_grade}", fontsize=7, pad=2)
        axes[r, 0].axis("off")

        # Columns 1–4: one per task
        for c, t in enumerate(tasks, start=1):
            cam_map = cam_extractor(tensor, task=t, class_idx=None)
            overlay = overlay_heatmap(rgb, cam_map, alpha=cfg.gradcam_alpha)

            with torch.no_grad():
                logits = model(tensor)
            pred = int(logits[t].argmax(dim=1).item())

            axes[r, c].imshow(overlay)
            axes[r, c].set_title(
                f"{task_names[t]}\npred={pred}", fontsize=7, pad=2
            )
            axes[r, c].axis("off")

    plt.suptitle(
        "GradCAM — All Classification Heads", fontsize=11, y=1.002
    )
    plt.tight_layout()

    out_path = os.path.join(cfg.output_dir, "gradcam_multihead.png")
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()

    cam_extractor.remove_hooks()
    print(f"[GradCAM] Multi-head comparison saved → {out_path}")


# =============================================================================
# SECTION 9 — VISUALIZATION & OUTPUT
# =============================================================================

def plot_training_curves(log_csv: str, output_dir: str) -> None:
    try:
        df = pd.read_csv(log_csv)
    except Exception as e:
        print(f"[WARNING] Could not read log CSV for plotting: {e}")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    stages = sorted(df["stage"].unique())
    colors = plt.cm.tab10(np.linspace(0, 0.5, len(stages)))

    row_offset = 0
    for i, stage in enumerate(stages):
        sub = df[df["stage"] == stage].reset_index(drop=True)
        x   = np.arange(row_offset + 1, row_offset + len(sub) + 1)
        axes[0].plot(x, sub["train_loss"], color=colors[i],
                     linestyle="-",  label=f"Stage {stage} train")
        axes[0].plot(x, sub["val_loss"],   color=colors[i],
                     linestyle="--", label=f"Stage {stage} val")
        if i > 0:
            axes[0].axvline(x=row_offset + 1, color="grey", alpha=0.4, linestyle=":")
        row_offset += len(sub)

    axes[0].set_title("Loss Curves")
    axes[0].set_xlabel("Epoch (global)")
    axes[0].set_ylabel("Loss")
    axes[0].legend(fontsize=7)
    axes[0].grid(alpha=0.3)

    row_offset = 0
    for i, stage in enumerate(stages):
        sub = df[df["stage"] == stage].reset_index(drop=True)
        x   = np.arange(row_offset + 1, row_offset + len(sub) + 1)
        axes[1].plot(x, sub["val_qwk"], color=colors[i],
                     marker="o", markersize=3, label=f"Stage {stage}")
        if i > 0:
            axes[1].axvline(x=row_offset + 1, color="grey", alpha=0.4, linestyle=":")
        row_offset += len(sub)

    axes[1].set_title("Validation Quadratic Weighted Kappa")
    axes[1].set_xlabel("Epoch (global)")
    axes[1].set_ylabel("QWK")
    axes[1].legend(fontsize=7)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(output_dir, "training_curves.png")
    plt.savefig(save_path, dpi=120)
    plt.close()
    print(f"[INFO] Training curves saved to {save_path}")


def plot_confusion_matrix(cm: np.ndarray, output_dir: str) -> None:
    fig, ax = plt.subplots(figsize=(max(6, len(cm)), max(5, len(cm))))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=[f"Pred {i}" for i in range(len(cm))],
        yticklabels=[f"True {i}" for i in range(len(cm))],
        ax=ax,
    )
    ax.set_title("Confusion Matrix — International DR Grade (TTA)")
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    save_path = os.path.join(output_dir, "confusion_matrix.png")
    plt.savefig(save_path, dpi=120)
    plt.close()
    print(f"[INFO] Confusion matrix saved to {save_path}")


# =============================================================================
# SECTION 10 — MAIN
# =============================================================================

def main() -> None:
    os.makedirs(CFG.output_dir, exist_ok=True)

    # ── Load data ─────────────────────────────────────────────────────────────
    df = load_csv(CFG)
    num_classes = infer_num_classes(df, CFG)
    train_df, val_df = stratified_split(df, CFG)

    # ── Build model ───────────────────────────────────────────────────────────
    print(f"\n[INFO] Building model: {CFG.model_name}")
    model = MultiTaskDRModel(
        model_name=CFG.model_name,
        num_classes=num_classes,
        dropout=CFG.dropout,
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"[INFO] Total parameters: {total_params:,}")

    # ── Build per-head class weights ──────────────────────────────────────────
    class_weights = {
        "intl": compute_class_weights_tensor(df, CFG.col_intl,  num_classes["intl"],  DEVICE),
        "aao":  compute_class_weights_tensor(df, CFG.col_aao,   num_classes["aao"],   DEVICE),
        "scot": compute_class_weights_tensor(df, CFG.col_scot,  num_classes["scot"],  DEVICE),
        "eye":  compute_class_weights_tensor(df, CFG.col_eye_side, num_classes["eye"], DEVICE),
    }

    # ── Build loss ────────────────────────────────────────────────────────────
    criterion = MultiTaskLoss(
        class_weights=class_weights,
        label_smoothing=CFG.label_smoothing,
        loss_weights=(CFG.loss_w_intl, CFG.loss_w_aao, CFG.loss_w_scot, CFG.loss_w_eye),
    ).to(DEVICE)

    # ── Progressive training ──────────────────────────────────────────────────
    run_all_stages(model, train_df, val_df, criterion, CFG)

    # ── Load best checkpoint ──────────────────────────────────────────────────
    best_ckpt_path = os.path.join(CFG.output_dir, CFG.best_ckpt)
    if os.path.exists(best_ckpt_path):
        model.load_state_dict(
            torch.load(best_ckpt_path, map_location=DEVICE, weights_only=True)
        )
        print(f"\n[INFO] Loaded best checkpoint from {best_ckpt_path}")

    # ── TTA + Final evaluation ────────────────────────────────────────────────
    final_metrics = final_evaluation(model, val_df, CFG, DEVICE)

    # ── Standard plots ────────────────────────────────────────────────────────
    log_path = os.path.join(CFG.output_dir, CFG.log_csv)
    plot_training_curves(log_path, CFG.output_dir)
    plot_confusion_matrix(final_metrics["cm"], CFG.output_dir)

    # ── GradCAM heatmaps ──────────────────────────────────────────────────────
    # 1. Primary-task grid: n_samples images × 3 panels (orig | cam | overlay)
    generate_gradcam_heatmaps(
        model=model,
        val_df=val_df,
        cfg=CFG,
        device=DEVICE,
    )

    # 2. Multi-head comparison: same images explained by all 4 heads
    gradcam_all_heads(
        model=model,
        val_df=val_df,
        cfg=CFG,
        device=DEVICE,
        n_samples=4,
    )

    # ── Final summary ─────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print(" TRAINING COMPLETE — SUMMARY")
    print("="*60)
    print(f"  Model        : {CFG.model_name}")
    print(f"  Best val QWK : {final_metrics['qwk']:.4f}")
    print(f"  Val Accuracy : {final_metrics['accuracy']:.4f}")
    print(f"  Val Macro F1 : {final_metrics['f1']:.4f}")
    print(f"  Checkpoint   : {best_ckpt_path}")
    print(f"  Log CSV      : {log_path}")
    print(f"  Curves PNG   : {os.path.join(CFG.output_dir, 'training_curves.png')}")
    print(f"  CM PNG       : {os.path.join(CFG.output_dir, 'confusion_matrix.png')}")
    print(f"  GradCAM grid : {os.path.join(CFG.output_dir, 'gradcam_grid.png')}")
    print(f"  Multi-head   : {os.path.join(CFG.output_dir, 'gradcam_multihead.png')}")
    print(f"  Per-image    : {os.path.join(CFG.output_dir, 'gradcam/')}")
    print("="*60)


if __name__ == "__main__":
    main()

[INFO] Using device: cuda
[INFO] GPU: Tesla P100-PCIE-16GB
[INFO] VRAM: 17.1 GB
[INFO] Dataset size after cleaning: 1151
  DR_grade_International_Clinical_DR_Severity_Scale: classes = [0, 1, 2, 3, 4]
  DR_grade_American_Academy_of_Ophthalmology: classes = [0, 1, 2, 3, 4]
  DR_grade_Scottish_DR_grading_protocol: classes = [0, 1, 2, 3, 4]
  left_versus_right_eye(left_0_right_1): classes = [0, 1]
[INFO] Inferred num_classes: {'intl': 5, 'aao': 5, 'scot': 5, 'eye': 2}
[INFO] Train: 920 | Val: 231

[INFO] Building model: tf_efficientnetv2_s.in21k_ft_in1k


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

[INFO] Total parameters: 20,201,825

 STAGE 1 | res=224 | mode=freeze | epochs=5 | lr=0.001
[INFO] Encoder frozen.
  Epoch 01/5 | lr=9.14e-04 | train_loss=1.6588 | val_loss=1.3421 | QWK=0.7921 | acc=0.5671 | F1=0.4552 | aao_acc=0.697 | scot_acc=0.636
  [Checkpoint] Best QWK = 0.7921 saved.
  Epoch 02/5 | lr=6.89e-04 | train_loss=1.5393 | val_loss=1.3842 | QWK=0.7578 | acc=0.6494 | F1=0.4202 | aao_acc=0.662 | scot_acc=0.667
  [EarlyStopping] 1/5 (best=0.7921)
  Epoch 03/5 | lr=4.11e-04 | train_loss=1.4374 | val_loss=1.2586 | QWK=0.6987 | acc=0.4719 | F1=0.3958 | aao_acc=0.537 | scot_acc=0.442
  [EarlyStopping] 2/5 (best=0.7921)
  Epoch 04/5 | lr=1.86e-04 | train_loss=1.3633 | val_loss=1.2707 | QWK=0.6966 | acc=0.5628 | F1=0.4124 | aao_acc=0.584 | scot_acc=0.524
  [EarlyStopping] 3/5 (best=0.7921)
  Epoch 05/5 | lr=1.00e-04 | train_loss=1.3540 | val_loss=1.3322 | QWK=0.7361 | acc=0.6320 | F1=0.4459 | aao_acc=0.684 | scot_acc=0.636
  [EarlyStopping] 4/5 (best=0.7921)
[INFO] Restored best 